# 🪢 Logistic Regression
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> The go-to model for binary classification. Logistic regression models the *probability* of an outcome — not just yes/no, but how confident the model is.

### What you'll learn
- Why linear regression fails for classification
- The logistic (sigmoid) function and log-odds
- Interpreting coefficients as odds ratios
- Confusion matrix, precision, recall, ROC/AUC
- Multinomial logistic regression (3+ classes)

### Dataset: Titanic (survived/not) + Default (credit card default)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Default

In [ ]:
import pandas as pd
default = pd.read_csv(f'{DATA_DIR}/Default.csv')
default['default_num'] = (default['default'] == 'Yes').astype(int)
default['student_num'] = (default['student'] == 'Yes').astype(int)
print(f"Default: {default.shape}  |  Default rate: {default['default_num'].mean():.2%}")
default.head()

## 📐 Part 1 — Why Not Linear Regression?

For a binary outcome Y ∈ {0,1}, a linear model can predict probabilities outside [0,1].  
We need a function that maps any real number to (0,1) — the **sigmoid (logistic) function**:

```
P(Y=1|X) = 1 / (1 + e^{-(β₀ + β₁X)})
```

Taking the log of the odds (logit transform) gives back a linear model:
```
log(P/(1-P)) = β₀ + β₁X
```

This is why it's called *logistic* regression — the log-odds are linear in X.

In [ ]:
# Show why linear regression fails for binary outcome
np.random.seed(42)
# Simulate: probability of default increases with balance
balance = np.sort(np.random.uniform(0, 2600, 400))
true_prob = 1/(1 + np.exp(-(balance - 1500)/300))
y_bin = (np.random.random(400) < true_prob).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Linear regression on binary outcome
from sklearn.linear_model import LinearRegression
lr_bad = LinearRegression().fit(balance.reshape(-1,1), y_bin)
y_linear = lr_bad.predict(balance.reshape(-1,1))

axes[0].scatter(balance, y_bin, alpha=0.15, color='#888', s=15)
axes[0].plot(balance, y_linear, color='#e85d20', lw=2.5, label='Linear regression')
axes[0].axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
axes[0].axhline(1, color='black', lw=0.8, ls='--', alpha=0.5)
axes[0].fill_between(balance, 0, 1, alpha=0.05, color='#1a7a45', label='Valid probability range')
axes[0].set_title('Linear Regression — predicts impossible probabilities!')
axes[0].set_xlabel('Balance ($)'); axes[0].set_ylabel('P(Default)')
axes[0].legend()

# Logistic regression — stays in [0,1]
log_reg = LogisticRegression().fit(balance.reshape(-1,1), y_bin)
y_logistic = log_reg.predict_proba(balance.reshape(-1,1))[:,1]

axes[1].scatter(balance, y_bin, alpha=0.15, color='#888', s=15)
axes[1].plot(balance, y_logistic, color='#1e5fa8', lw=2.5, label='Logistic regression')
axes[1].axhline(0.5, color='#e85d20', lw=1.5, ls='--', alpha=0.7, label='Decision boundary (p=0.5)')
axes[1].set_title('Logistic Regression — always in [0,1] ✓')
axes[1].set_xlabel('Balance ($)'); axes[1].set_ylabel('P(Default)')
axes[1].legend()

plt.tight_layout()
plt.show()
print("📌 Logistic regression always produces valid probabilities between 0 and 1.")

In [ ]:
# Fit and interpret with statsmodels (shows coefficients, p-values)
X_def = sm.add_constant(default[['balance','income','student_num']])
logit_model = sm.Logit(default['default_num'], X_def).fit()
print(logit_model.summary())

print("\n=== INTERPRETING COEFFICIENTS ===")
coefs = logit_model.params
print(f"\nbalance coef = {coefs['balance']:.6f}")
print(f"  → each $1 increase in balance multiplies odds of default by {np.exp(coefs['balance']):.4f}")
print(f"  → each $100 increase → odds × {np.exp(coefs['balance']*100):.3f}")
print(f"\nstudent_num coef = {coefs['student_num']:.4f}")
print(f"  → being a student multiplies odds of default by {np.exp(coefs['student_num']):.3f}")
print(f"  → students have {np.exp(coefs['student_num']):.2f}x the odds of default (vs non-students)")

In [ ]:
# Train model and evaluate
X = default[['balance','income','student_num']].values
y = default['default_num'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

clf = LogisticRegression(random_state=42).fit(X_tr_s, y_tr)
y_pred = clf.predict(X_te_s)
y_prob = clf.predict_proba(X_te_s)[:,1]

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_te, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Default','Default'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix\n(threshold = 0.5)')

# ROC Curve
fpr, tpr, _ = roc_curve(y_te, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#1e5fa8', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random (AUC=0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#1e5fa8')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n=== Classification Report ===")
print(classification_report(y_te, y_pred, target_names=['No Default','Default']))
print(f"AUC-ROC: {roc_auc:.4f}")
print("\n📌 High accuracy (97%) but low recall on defaults — class imbalance issue!")
print("   Only 3.3% of customers default, so predicting 'no default' always gets 97% accuracy.")

In [ ]:
# Decision boundary visualization
fig, ax = plt.subplots(figsize=(9, 5))
xx = np.linspace(0, 2700, 300)
thresholds = [0.3, 0.5, 0.7]
colors_t = ['#e85d20','#1e5fa8','#1a7a45']

# Use only balance for clean 2D viz
clf_1d = LogisticRegression().fit(
    scaler.fit_transform(default[['balance']]), 
    default['default_num']
)
x_range = np.linspace(default['balance'].min(), default['balance'].max(), 300).reshape(-1,1)
probs = clf_1d.predict_proba(scaler.transform(x_range))[:,1]

ax.scatter(default.loc[default['default_num']==0,'balance'], 
           np.zeros(sum(default['default_num']==0)), alpha=0.05, color='#1e5fa8', s=5)
ax.scatter(default.loc[default['default_num']==1,'balance'], 
           np.ones(sum(default['default_num']==1)), alpha=0.2, color='#e85d20', s=5)
ax.plot(x_range, probs, color='#1a7a45', lw=2.5, label='P(Default | Balance)')

for t, c in zip(thresholds, colors_t):
    ax.axhline(t, color=c, lw=1.2, ls='--', alpha=0.8, label=f'Threshold = {t}')

ax.set_xlabel('Credit Card Balance ($)')
ax.set_ylabel('Estimated P(Default)')
ax.set_title('Logistic Regression: Probability of Default vs Balance')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print("\n📌 Lowering the threshold catches more actual defaults but flags more non-defaulters.")
print("   The right threshold depends on the cost of false positives vs false negatives.")

In [ ]:
answers = {
    "q1": "",  # Why does logistic regression use the sigmoid function instead of a linear model?
    "q2": "",  # A coefficient of 0.003 on 'balance' means what in terms of odds?
    "q3": "",  # What does AUC-ROC = 0.5 mean?
    "q4": "",  # When would you lower the classification threshold below 0.5?
    "q5": "",  # What is the difference between accuracy and recall?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Logistic Regression"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()
